# KV Activation Visualization

기존 `run_model.py --collect_qkv`로 저장한 KV activation을 탐색하는 노트북입니다.

기본 저장 경로는 `activations/<model_dir>/*_{raw,rot}.pt`이며, tensor는 저장 직후 shape `(L, B, H, S, D)`를 `(L, H, B*S, D)`로 변환해 시각화합니다.

## 0. Activation 수집 예시

아직 activation 파일이 없다면 레포 루트에서 아래처럼 먼저 수집합니다.

```bash
python run_model.py --model meta-llama/Llama-2-7b-hf --collect_qkv --device cuda

# rotation 적용본도 같이 보고 싶으면
python run_model.py --model meta-llama/Llama-2-7b-hf --collect_qkv --rotate hadamard --device cuda
```

LLaMA 계열은 `k`, `v`, `k_rope`, `q_rope`, `k_grad`, `v_grad`가 저장될 수 있고, OPT 계열은 보통 `k`, `v`, `k_grad`, `v_grad`가 저장됩니다.

In [ ]:
from pathlib import Path
import math
import os

import torch
import numpy as np
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="notebook")
except Exception:
    sns = None
    plt.style.use("default")

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

ACT_DIR = REPO_ROOT / "activations"
print(f"repo root: {REPO_ROOT}")
print(f"activation dir: {ACT_DIR}")

## 1. 설정

`MODEL_ID`는 Hugging Face model id 또는 저장 디렉터리명입니다. 예: `meta-llama/Llama-2-7b-hf`는 `meta-llama_Llama-2-7b-hf`로 저장됩니다.

In [ ]:
MODEL_ID = "meta-llama/Llama-2-7b-hf"
ROT_TAG = "raw"  # "raw" 또는 "rot"

# 자주 바꾸는 plot 대상
KIND = "k"       # "k", "v", "k_rope", "q_rope", "k_grad", "v_grad"
LAYER_IDX = 0
HEAD_IDX = 0

# 긴 sequence를 모두 그리면 느릴 수 있어 앞쪽 일부만 heatmap에 사용합니다.
MAX_TOKENS_FOR_HEATMAP = 256

# raw/rot histogram 비교에 사용할 랜덤 샘플 수입니다.
HIST_SAMPLE_SIZE = 200_000
HIST_SAMPLE_SEED = 0

# None이면 전체 layer/head에서 샘플링합니다. 특정 부분만 보려면 정수로 바꾸세요.
HIST_LAYER_IDX = None  # 예: 0
HIST_HEAD_IDX = None   # 예: 0

# hidden dimension index별 raw/rot 값 범위 plot 설정입니다.
HIDDEN_DIM_LAYER_IDX = 0
HIDDEN_DIM_TOKEN_SAMPLE_SIZE = 4096  # None이면 모든 token/sample 사용
HIDDEN_DIM_Y_LIMIT = None            # 예: (-0.5, 0.5)
OUTLIER_PERCENTILES = [99.0, 99.9, 99.99]
OUTLIER_TOPK_DIMS = 20

model_dir = MODEL_ID.replace("/", "_")
act_root = ACT_DIR / model_dir
act_root

In [ ]:
def list_activation_sets(act_dir: Path = ACT_DIR):
    if not act_dir.exists():
        return []
    rows = []
    for model_path in sorted(p for p in act_dir.iterdir() if p.is_dir()):
        files = sorted(model_path.glob("*.pt"))
        tags = sorted({p.stem.rsplit("_", 1)[-1] for p in files if p.stem.endswith(("_raw", "_rot"))})
        kinds = sorted({p.stem.rsplit("_", 1)[0] for p in files if p.stem.endswith(("_raw", "_rot"))})
        rows.append({"model_dir": model_path.name, "tags": tags, "kinds": kinds, "n_files": len(files)})
    return rows

available = list_activation_sets()
if not available:
    print("No activation files found. Run `python run_model.py --collect_qkv ...` first.")
else:
    for row in available:
        print(f"{row['model_dir']} | tags={row['tags']} | kinds={row['kinds']} | files={row['n_files']}")

## 2. 로드 유틸리티

`run_model.py`가 저장한 `(L, B, H, S, D)` 텐서를 plotting에 편한 `(L, H, T, D)`로 변환합니다. 이미 `(L, H, T, D)`인 텐서도 그대로 처리합니다.

In [ ]:
def activation_path(kind: str, tag: str = ROT_TAG, root: Path = act_root) -> Path:
    return root / f"{kind}_{tag}.pt"

def to_layer_head_time_dim(x: torch.Tensor) -> torch.Tensor:
    """Normalize saved activations to (layers, heads, tokens, head_dim)."""
    if x.ndim == 5:
        # (L, B, H, S, D) -> (L, H, B*S, D)
        l, b, h, s, d = x.shape
        return x.transpose(1, 2).reshape(l, h, b * s, d)
    if x.ndim == 4:
        return x
    raise ValueError(f"Expected 4D or 5D tensor, got shape={tuple(x.shape)}")

def load_activation(kind: str, tag: str = ROT_TAG, root: Path = act_root) -> torch.Tensor:
    path = activation_path(kind, tag, root)
    if not path.exists():
        raise FileNotFoundError(f"Missing activation file: {path}")
    x = torch.load(path, map_location="cpu")
    x = to_layer_head_time_dim(x).float()
    print(f"loaded {path.name}: shape={tuple(x.shape)}, dtype={x.dtype}")
    return x

def available_kinds(tag: str = ROT_TAG, root: Path = act_root):
    if not root.exists():
        return []
    suffix = f"_{tag}.pt"
    return sorted(p.name[:-len(suffix)] for p in root.glob(f"*{suffix}"))

print("available kinds:", available_kinds())

In [ ]:
x = load_activation(KIND, ROT_TAG)
n_layers, n_heads, n_tokens, head_dim = x.shape
print({"layers": n_layers, "heads": n_heads, "tokens": n_tokens, "head_dim": head_dim})

## 3. 단일 Layer/Head Heatmap

행은 token/time 축, 열은 head dimension입니다. 값 범위가 큰 경우 percentile clipping을 적용해 패턴이 더 잘 보이게 합니다.

In [ ]:
def plot_head_heatmap(x: torch.Tensor, layer: int = LAYER_IDX, head: int = HEAD_IDX,
                      max_tokens: int = MAX_TOKENS_FOR_HEATMAP, clip_percentile: float = 99.0,
                      title: str | None = None):
    data = x[layer, head, :max_tokens].numpy()
    vmax = np.percentile(np.abs(data), clip_percentile)
    vmax = float(vmax) if vmax > 0 else None

    fig, ax = plt.subplots(figsize=(12, 5))
    im = ax.imshow(data, aspect="auto", cmap="coolwarm", vmin=None if vmax is None else -vmax, vmax=vmax)
    ax.set_xlabel("head dimension")
    ax.set_ylabel("token")
    ax.set_title(title or f"{KIND}_{ROT_TAG} | layer={layer}, head={head}")
    fig.colorbar(im, ax=ax, label="activation")
    plt.tight_layout()

plot_head_heatmap(x)

## 4. Layer/Head별 Activation 크기

각 layer/head에 대해 전체 token과 channel의 RMS 또는 max-abs를 요약합니다.

In [ ]:
def summarize_layer_head(x: torch.Tensor):
    rms = x.pow(2).mean(dim=(-1, -2)).sqrt()
    max_abs = x.abs().amax(dim=(-1, -2))
    mean_abs = x.abs().mean(dim=(-1, -2))
    return {"rms": rms, "max_abs": max_abs, "mean_abs": mean_abs}

summary = summarize_layer_head(x)

def plot_layer_head_metric(metric: torch.Tensor, name: str):
    fig, ax = plt.subplots(figsize=(12, 5))
    im = ax.imshow(metric.numpy(), aspect="auto", cmap="viridis")
    ax.set_xlabel("head")
    ax.set_ylabel("layer")
    ax.set_title(f"{KIND}_{ROT_TAG} {name} by layer/head")
    fig.colorbar(im, ax=ax, label=name)
    plt.tight_layout()

plot_layer_head_metric(summary["rms"], "RMS")
plot_layer_head_metric(summary["max_abs"], "max |x|")

In [ ]:
def plot_layer_metric_lines(summary: dict[str, torch.Tensor]):
    fig, ax = plt.subplots(figsize=(12, 4))
    for name, metric in summary.items():
        ax.plot(metric.mean(dim=1).numpy(), marker="o", label=f"mean head {name}")
    ax.set_xlabel("layer")
    ax.set_ylabel("value")
    ax.set_title(f"{KIND}_{ROT_TAG} layer-wise summary")
    ax.legend()
    plt.tight_layout()

plot_layer_metric_lines(summary)

## 5. Hidden Dimension별 Raw vs Rot 값 범위

선택한 layer에서 head와 head dimension을 `hidden dimension index`로 펼친 뒤, token/sample 축에 대해 min/max와 percentile band를 그립니다. 참고 그림처럼 dimension별 outlier가 rotation 후 얼마나 줄었는지 보기 위한 plot입니다.

In [ ]:
def maybe_load_pair(kind: str, root: Path = act_root):
    raw_path = activation_path(kind, "raw", root)
    rot_path = activation_path(kind, "rot", root)
    if not raw_path.exists() or not rot_path.exists():
        print(f"Need both {raw_path.name} and {rot_path.name} for comparison.")
        return None, None
    return load_activation(kind, "raw", root), load_activation(kind, "rot", root)

def layer_to_token_hidden(x: torch.Tensor, layer: int = HIDDEN_DIM_LAYER_IDX,
                          token_sample_size: int | None = HIDDEN_DIM_TOKEN_SAMPLE_SIZE,
                          seed: int = HIST_SAMPLE_SEED) -> torch.Tensor:
    # (L, H, T, D) -> selected layer -> (T, H*D)
    data = x[layer].permute(1, 0, 2).reshape(x.shape[2], -1)
    if token_sample_size is not None and data.shape[0] > token_sample_size:
        g = torch.Generator().manual_seed(seed)
        idx = torch.randperm(data.shape[0], generator=g)[:token_sample_size]
        data = data[idx]
    return data

def hidden_dim_stats(data: torch.Tensor) -> dict[str, np.ndarray]:
    qs = torch.quantile(data, torch.tensor([0.01, 0.25, 0.75, 0.99], dtype=data.dtype), dim=0)
    return {
        "min": data.min(dim=0).values.numpy(),
        "max": data.max(dim=0).values.numpy(),
        "p01": qs[0].numpy(),
        "p25": qs[1].numpy(),
        "p75": qs[2].numpy(),
        "p99": qs[3].numpy(),
    }

def plot_hidden_dim_range(kind: str = KIND, layer: int = HIDDEN_DIM_LAYER_IDX):
    raw_x, rot_x = maybe_load_pair(kind)
    if raw_x is None:
        return

    raw_data = layer_to_token_hidden(raw_x, layer=layer, seed=HIST_SAMPLE_SEED)
    rot_data = layer_to_token_hidden(rot_x, layer=layer, seed=HIST_SAMPLE_SEED)
    raw_stats = hidden_dim_stats(raw_data)
    rot_stats = hidden_dim_stats(rot_data)
    xs = np.arange(raw_data.shape[1])

    fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), sharey=True)
    for ax, stats, title in [
        (axes[0], raw_stats, "Before rotation (raw)"),
        (axes[1], rot_stats, "With rotation (rot)"),
    ]:
        ax.plot(xs, stats["min"], color="#1e90ff", linewidth=0.7, label="Min/Max")
        ax.plot(xs, stats["max"], color="#1e90ff", linewidth=0.7)
        ax.plot(xs, stats["p01"], color="#e83e8c", linewidth=0.9, label="1/99 Percentile")
        ax.plot(xs, stats["p99"], color="#e83e8c", linewidth=0.9)
        ax.plot(xs, stats["p25"], color="#ffcc66", linewidth=1.0, label="25/75 Percentile")
        ax.plot(xs, stats["p75"], color="#ffcc66", linewidth=1.0)
        ax.set_title(title, fontsize=16, fontweight="bold")
        ax.set_xlabel("Hidden dimension index", fontsize=13)
        ax.grid(False)
        if HIDDEN_DIM_Y_LIMIT is not None:
            ax.set_ylim(*HIDDEN_DIM_Y_LIMIT)
    axes[0].set_ylabel("Activation value", fontsize=13)
    axes[1].legend(loc="upper left", frameon=True)
    fig.suptitle(f"{kind} activation range by hidden dimension | layer={layer} | samples={raw_data.shape[0]:,}")
    plt.tight_layout()
    return raw_stats, rot_stats

hidden_dim_stats_pair = plot_hidden_dim_range(KIND, HIDDEN_DIM_LAYER_IDX)

## 6. Outlier 감소 정량 확인

raw activation에서 percentile threshold를 잡고, 같은 threshold를 rot activation에 적용해 초과 비율이 줄었는지 확인합니다. `max |x|`, high percentile, threshold 초과 개수/비율이 함께 줄면 rotation이 outlier를 완화했다고 볼 수 있습니다.

In [ ]:
def outlier_report(kind: str = KIND, layer: int = HIDDEN_DIM_LAYER_IDX,
                   percentiles: list[float] = OUTLIER_PERCENTILES):
    raw_x, rot_x = maybe_load_pair(kind)
    if raw_x is None:
        return None

    raw_data = layer_to_token_hidden(raw_x, layer=layer, seed=HIST_SAMPLE_SEED).abs()
    rot_data = layer_to_token_hidden(rot_x, layer=layer, seed=HIST_SAMPLE_SEED).abs()
    raw_flat = raw_data.reshape(-1)
    rot_flat = rot_data.reshape(-1)

    rows = []
    for pct in percentiles:
        threshold = torch.quantile(raw_flat, pct / 100.0)
        raw_rate = (raw_flat > threshold).float().mean().item()
        rot_rate = (rot_flat > threshold).float().mean().item()
        rows.append({
            "raw_percentile_threshold": pct,
            "threshold_from_raw": threshold.item(),
            "raw_exceed_rate": raw_rate,
            "rot_exceed_rate": rot_rate,
            "rot/raw_exceed_rate": rot_rate / max(raw_rate, 1e-12),
            "reduction_%": 100.0 * (1.0 - rot_rate / max(raw_rate, 1e-12)),
        })

    print(f"Outlier report for {kind} | layer={layer} | samples={raw_data.shape[0]:,} x hidden_dim={raw_data.shape[1]:,}")
    print(f"raw max |x|: {raw_flat.max().item():.6g}")
    print(f"rot max |x|: {rot_flat.max().item():.6g}")
    print(f"max |x| reduction: {100.0 * (1.0 - rot_flat.max().item() / max(raw_flat.max().item(), 1e-12)):.2f}%")
    print()
    for row in rows:
        print(
            f"raw p{row['raw_percentile_threshold']:g} threshold={row['threshold_from_raw']:.6g} | "
            f"raw exceed={row['raw_exceed_rate']:.6%}, rot exceed={row['rot_exceed_rate']:.6%}, "
            f"reduction={row['reduction_%']:.2f}%"
        )
    return rows, raw_data, rot_data

outlier_result = outlier_report(KIND, HIDDEN_DIM_LAYER_IDX)

In [ ]:
def plot_outlier_reduction_by_dim(outlier_result, topk: int = OUTLIER_TOPK_DIMS):
    if outlier_result is None:
        return
    _, raw_data, rot_data = outlier_result
    raw_max_by_dim = raw_data.max(dim=0).values
    rot_max_by_dim = rot_data.max(dim=0).values
    ratio = rot_max_by_dim / raw_max_by_dim.clamp_min(1e-12)
    top_idx = torch.topk(raw_max_by_dim, k=min(topk, raw_max_by_dim.numel())).indices

    xs = np.arange(len(top_idx))
    width = 0.38
    fig, ax = plt.subplots(figsize=(13, 4.5))
    ax.bar(xs - width / 2, raw_max_by_dim[top_idx].numpy(), width=width, label="raw max |x|", color="#4c78a8")
    ax.bar(xs + width / 2, rot_max_by_dim[top_idx].numpy(), width=width, label="rot max |x|", color="#f58518")
    ax.set_xticks(xs)
    ax.set_xticklabels([str(i.item()) for i in top_idx], rotation=45, ha="right")
    ax.set_xlabel("hidden dimension index, sorted by raw max |x|")
    ax.set_ylabel("max |activation|")
    ax.set_title(f"Top-{len(top_idx)} raw outlier dimensions: raw vs rot")
    ax.legend()
    plt.tight_layout()

    print("Top raw-outlier dimensions where rot/raw max |x| is largest:")
    worst = torch.topk(ratio[top_idx], k=min(5, len(top_idx))).indices
    for local_idx in worst.tolist():
        dim = top_idx[local_idx].item()
        print(f"  dim={dim:5d} raw={raw_max_by_dim[dim].item():.6g} rot={rot_max_by_dim[dim].item():.6g} ratio={ratio[dim].item():.3f}")

plot_outlier_reduction_by_dim(outlier_result)

## 7. Raw vs Rot Distribution 샘플 비교

전체 tensor가 크므로 `HIST_SAMPLE_SIZE`개만 랜덤 샘플링해서 `raw`와 `rot`의 histogram을 따로 보여줍니다. 단, 비교가 쉽도록 같은 bin/range를 사용합니다. `HIST_LAYER_IDX`, `HIST_HEAD_IDX`를 지정하면 특정 layer/head 안에서만 샘플링합니다.

In [ ]:
def maybe_load_pair(kind: str, root: Path = act_root):
    raw_path = activation_path(kind, "raw", root)
    rot_path = activation_path(kind, "rot", root)
    if not raw_path.exists() or not rot_path.exists():
        print(f"Need both {raw_path.name} and {rot_path.name} for comparison.")
        return None, None
    return load_activation(kind, "raw", root), load_activation(kind, "rot", root)

def select_hist_scope(x: torch.Tensor, layer: int | None = HIST_LAYER_IDX, head: int | None = HIST_HEAD_IDX):
    scoped = x
    scope = []
    if layer is not None:
        scoped = scoped[layer:layer + 1]
        scope.append(f"layer={layer}")
    if head is not None:
        scoped = scoped[:, head:head + 1]
        scope.append(f"head={head}")
    return scoped, ", ".join(scope) if scope else "all layers/heads"

def sample_flat(x: torch.Tensor, n: int = HIST_SAMPLE_SIZE, seed: int = HIST_SAMPLE_SEED):
    flat = x.reshape(-1)
    if flat.numel() <= n:
        return flat.numpy()
    g = torch.Generator().manual_seed(seed)
    idx = torch.randint(flat.numel(), (n,), generator=g)
    return flat[idx].numpy()

def plot_raw_rot_histograms(kind: str = KIND, sample_size: int = HIST_SAMPLE_SIZE,
                            bins: int = 160, clip_percentile: float = 99.9):
    raw_x, rot_x = maybe_load_pair(kind)
    if raw_x is None:
        return

    raw_scoped, scope = select_hist_scope(raw_x)
    rot_scoped, _ = select_hist_scope(rot_x)
    raw_sample = sample_flat(raw_scoped, n=sample_size, seed=HIST_SAMPLE_SEED)
    rot_sample = sample_flat(rot_scoped, n=sample_size, seed=HIST_SAMPLE_SEED + 1)

    signed_limit = np.percentile(np.abs(np.concatenate([raw_sample, rot_sample])), clip_percentile)
    signed_limit = float(signed_limit) if signed_limit > 0 else 1.0
    signed_range = (-signed_limit, signed_limit)
    abs_limit = signed_limit

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex="col")
    axes[0, 0].hist(raw_sample, bins=bins, range=signed_range, density=True, color="#4c78a8", alpha=0.85)
    axes[0, 0].set_title("raw signed activation")
    axes[0, 0].set_ylabel("density")

    axes[1, 0].hist(rot_sample, bins=bins, range=signed_range, density=True, color="#f58518", alpha=0.85)
    axes[1, 0].set_title("rot signed activation")
    axes[1, 0].set_xlabel("value")
    axes[1, 0].set_ylabel("density")

    axes[0, 1].hist(np.abs(raw_sample), bins=bins, range=(0, abs_limit), density=True, color="#4c78a8", alpha=0.85)
    axes[0, 1].set_yscale("log")
    axes[0, 1].set_title("raw |activation|, log density")

    axes[1, 1].hist(np.abs(rot_sample), bins=bins, range=(0, abs_limit), density=True, color="#f58518", alpha=0.85)
    axes[1, 1].set_yscale("log")
    axes[1, 1].set_title("rot |activation|, log density")
    axes[1, 1].set_xlabel("|value|")

    fig.suptitle(f"{kind}: raw vs rot sampled distribution | {scope} | n={len(raw_sample):,}")
    plt.tight_layout()
    return raw_sample, rot_sample

hist_samples = plot_raw_rot_histograms(KIND)
if hist_samples is not None:
    raw_sample, rot_sample = hist_samples

In [ ]:
def print_sample_quantiles(values: np.ndarray, name: str = "activation"):
    vals = np.abs(values)
    qs = np.array([0.5, 0.9, 0.95, 0.99, 0.999])
    qv = np.quantile(vals, qs)
    print(name)
    for q, v in zip(qs.tolist(), qv.tolist()):
        print(f"  p{q * 100:05.2f}: {v:.6g}")
    print(f"  max  : {vals.max():.6g}")

if "raw_sample" in globals() and "rot_sample" in globals():
    print_sample_quantiles(raw_sample, f"{KIND}_raw sampled |x|")
    print_sample_quantiles(rot_sample, f"{KIND}_rot sampled |x|")

## 8. Token-wise Norm 변화

특정 layer/head에서 token 위치별 norm 변화를 봅니다. 긴 context에서 특정 구간에 outlier가 몰리는지 확인할 때 유용합니다.

In [ ]:
def plot_token_norm(x: torch.Tensor, layer: int = LAYER_IDX, head: int = HEAD_IDX):
    data = x[layer, head]
    l2 = data.pow(2).sum(dim=-1).sqrt().numpy()
    max_abs = data.abs().amax(dim=-1).numpy()

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(l2, label="L2 norm")
    ax.plot(max_abs, label="max |x|", alpha=0.8)
    ax.set_xlabel("token")
    ax.set_title(f"{KIND}_{ROT_TAG} token-wise norm | layer={layer}, head={head}")
    ax.legend()
    plt.tight_layout()

plot_token_norm(x)

## 9. Raw vs Rot RMS 비교

`raw`와 `rot` activation이 둘 다 있으면 layer/head별 RMS도 함께 비교합니다.

In [ ]:
raw_x, rot_x = maybe_load_pair(KIND)

if raw_x is not None:
    raw_rms = summarize_layer_head(raw_x)["rms"]
    rot_rms = summarize_layer_head(rot_x)["rms"]
    ratio = rot_rms / raw_rms.clamp_min(1e-12)

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, metric, title in zip(axes, [raw_rms, rot_rms, ratio], ["raw RMS", "rot RMS", "rot/raw RMS"]):
        im = ax.imshow(metric.numpy(), aspect="auto", cmap="viridis")
        ax.set_xlabel("head")
        ax.set_ylabel("layer")
        ax.set_title(title)
        fig.colorbar(im, ax=ax)
    plt.tight_layout()


## 10. K/V 한 번에 비교

`k`와 `v`를 같은 tag에서 비교합니다.

In [ ]:
def plot_kv_layer_summary(tag: str = ROT_TAG):
    tensors = {}
    for kind in ["k", "v", "k_rope"]:
        path = activation_path(kind, tag)
        if path.exists():
            tensors[kind] = load_activation(kind, tag)

    if not tensors:
        print(f"No k/v activation files found for tag={tag}")
        return

    fig, ax = plt.subplots(figsize=(12, 4))
    for kind, tensor in tensors.items():
        rms_by_layer = summarize_layer_head(tensor)["rms"].mean(dim=1).numpy()
        ax.plot(rms_by_layer, marker="o", label=kind)
    ax.set_xlabel("layer")
    ax.set_ylabel("mean RMS across heads")
    ax.set_title(f"KV activation RMS by layer ({tag})")
    ax.legend()
    plt.tight_layout()

plot_kv_layer_summary(ROT_TAG)